# Data quality and idempotency validation
Verifies row counts, checks that zero duplicate business keys exist across bronze, silver, and gold, confirms all mandatory data quality rules passed, and displays the final gold analysis.

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

In [0]:
# 1. Verify row counts and uniqueness (0 duplicates) across all layers
tables_to_check = [
    ("bronze.electricity_price_api_raw", ["bronze_record_id"]),
    ("bronze.weather_api_raw", ["bronze_record_id"]),
    ("silver.electricity_price_hourly", ["price_area", "time_start_utc"]),
    ("silver.weather_hourly", ["station_id", "time_start_utc"]),
    ("silver.weather_price_hourly", ["price_area", "station_id", "time_start_utc"]),
    ("gold.price_by_temperature_band", ["price_area", "temperature_band_order"])
]

summary_rows = []
for table_path, key_cols in tables_to_check:
    full_name = f"{CATALOG}.{table_path}"
    df = spark.table(full_name)
    total_rows = df.count()
    duplicate_keys = (
        df.groupBy(*key_cols).count().filter("count > 1").count()
    )
    summary_rows.append((table_path, total_rows, duplicate_keys))

    if total_rows == 0:
        raise AssertionError(f"Validation failed: {full_name} is empty")
    if duplicate_keys > 0:
        raise AssertionError(
            f"Validation failed: {full_name} has {duplicate_keys} duplicate key(s)"
        )

summary_df = spark.createDataFrame(
    summary_rows, ["table_name", "total_rows", "duplicate_key_count"]
)
display(summary_df)

In [0]:
# 2. Check operational Data Quality logs for any failed ERROR checks
dq_df = spark.table(f"{CATALOG}.ops.data_quality_results")
failed_errors = dq_df.filter(
    (F.col("severity") == "ERROR") & (F.col("passed") == False)
).count()

display(
    dq_df.orderBy(F.col("checked_at_utc").desc()).limit(20)
)

if failed_errors > 0:
    raise AssertionError(
        f"Validation failed: {failed_errors} ERROR-level data quality check(s) failed"
    )

print("All table uniqueness and data quality checks passed")

In [0]:
# 3. Display final Gold output for visualization
gold_df = (
    spark.table(f"{CATALOG}.gold.price_by_temperature_band")
    .orderBy("temperature_band_order")
)
display(gold_df)

Databricks visualization. Run in Databricks to view.